In [0]:
!pip install kaggle

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import os

os.environ["KAGGLE_USERNAME"] = ""
os.environ["KAGGLE_KEY"] = ""

print("Kaggle credentials configured!")

Kaggle credentials configured!


In [0]:
spark.sql("""
CREATE SCHEMA IF NOT EXISTS workspace.ecommerce
""")

DataFrame[]

In [0]:
spark.sql("""
CREATE VOLUME IF NOT EXISTS workspace.ecommerce.ecommerce_data
""")

DataFrame[]

In [0]:
%sh
cd /Volumes/workspace/ecommerce/ecommerce_data
kaggle datasets download -d mkechinov/ecommerce-behavior-data-from-multi-category-store

Dataset URL: https://www.kaggle.com/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store
License(s): copyright-authors


100%|██████████| 4.29G/4.29G [00:50<00:00, 90.5MB/s]


In [0]:
%sh
cd /Volumes/workspace/ecommerce/ecommerce_data
unzip -o ecommerce-behavior-data-from-multi-category-store.zip
ls -lh

Archive:  ecommerce-behavior-data-from-multi-category-store.zip
  inflating: 2019-Nov.csv            
  inflating: 2019-Oct.csv            
total 18G
-rwxrwxrwx 1 spark-fd5daa3f-4f51-4aab-adcc-2b nogroup 8.4G Jan 12 20:24 2019-Nov.csv
-rwxrwxrwx 1 spark-fd5daa3f-4f51-4aab-adcc-2b nogroup 5.3G Jan 12 20:26 2019-Oct.csv
drwxrwxrwx 2 nobody                           nogroup 4.0K Jan 12 19:41 delta
-rwxrwxrwx 1 spark-fd5daa3f-4f51-4aab-adcc-2b nogroup 4.3G Jan 12 20:23 ecommerce-behavior-data-from-multi-category-store.zip
drwxrwxrwx 2 nobody                           nogroup 4.0K Jan 12 19:41 outputs


In [0]:
%sh
cd /Volumes/workspace/ecommerce/ecommerce_data
rm -f ecommerce-behavior-data-from-multi-category-store.zip
ls -lh

total 14G
-rwxrwxrwx 1 spark-fd5daa3f-4f51-4aab-adcc-2b nogroup 8.4G Jan 12 20:24 2019-Nov.csv
-rwxrwxrwx 1 spark-fd5daa3f-4f51-4aab-adcc-2b nogroup 5.3G Jan 12 20:26 2019-Oct.csv
drwxrwxrwx 2 nobody                           nogroup 4.0K Jan 12 19:41 delta
drwxrwxrwx 2 nobody                           nogroup 4.0K Jan 12 19:41 outputs


In [0]:
%restart_python

In [0]:
from pyspark.sql import functions as F
csv_path = "/Volumes/workspace/ecommerce/ecommerce_data/2019-Oct.csv"
delta_path = "/Volumes/workspace/ecommerce/ecommerce_data/delta/events_oct2019"

db = "workspace.ecommerce"
table_managed = f"{db}.events_oct2019_managed"
table_external = f"{db}.events_oct2019_external"

events = (spark.read.format("csv")
          .option("header", "true")
          .option("inferSchema", "true")
          .load(csv_path))

events = (events
          .withColumn("price", F.col("price").cast("double")))

In [0]:
(events.write
 .format("delta")
 .mode("overwrite")
 .save(delta_path))

print("Delta written to:", delta_path)

Delta written to: /Volumes/workspace/ecommerce/ecommerce_data/delta/events_oct2019


In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {db}")

(events.write
 .format("delta")
 .mode("overwrite")
 .saveAsTable(table_managed))

display(spark.table(table_managed).limit(5))

event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
2019-10-05T16:36:18.000Z,view,43500001,2099306010899383229,null,crown,23.68,516651655,5092345c-59a9-4de8-b944-413aadbf6f93
2019-10-05T16:36:18.000Z,view,16600209,2053013560287560373,null,adil,114.55,543297573,afec0090-781c-41da-a19f-5d668283a901
2019-10-05T16:36:18.000Z,view,23300165,2053013561956893455,null,null,33.15,515441330,74de18dc-2534-4f23-aa7a-d0ca6d0a4d06
2019-10-05T16:36:18.000Z,view,4100214,2053013561218695907,null,nintendo,360.34,512973571,bf099e2d-28c7-4697-9846-b95c5cd562fa
2019-10-05T16:36:18.000Z,cart,1004768,2053013555631882655,electronics.smartphone,samsung,252.55,549499444,adfbaea3-4d07-46f7-b47b-888b7215146c


In [0]:
wrong_schema = (events.limit(10)
                .select(
                    "event_time","event_type","product_id","category_id","category_code","brand",
                    F.col("price").cast("string").alias("price"), 
                    "user_id","user_session"
                ))

try:
    (wrong_schema.write
     .format("delta")
     .mode("append")
     .saveAsTable(table_managed))
    print("Unexpected: append succeeded")
except Exception as e:
    print("Schema enforcement triggered (expected).")
    print(str(e)[:600])

Schema enforcement triggered (expected).
[DELTA_FAILED_TO_MERGE_FIELDS] Failed to merge fields 'price' and 'price'.

JVM stacktrace:
com.databricks.sql.transaction.tahoe.DeltaAnalysisException
	at com.databricks.sql.transaction.tahoe.schema.SchemaMergingUtils$.$anonfun$mergeDataTypes$1(SchemaMergingUtils.scala:231)
	at scala.collection.ArrayOps$.map$extension(ArrayOps.scala:936)
	at com.databricks.sql.transaction.tahoe.schema.SchemaMergingUtils$.merge$1(SchemaMergingUtils.scala:217)
	at com.databricks.sql.transaction.tahoe.schema.SchemaMergingUtils$.mergeDataTypes(SchemaMergingUtils.scala:335)
	at com.databricks.sql.transaction.tahoe


In [0]:
key_cols = ["user_session", "event_time", "event_type", "product_id"]

events_keyed = (events
    .withColumn("dedupe_key",
                F.sha2(F.concat_ws("||", *[F.col(c).cast("string") for c in key_cols]), 256))
)

target_keyed = f"{db}.events_oct2019_keyed"
(events_keyed.write.format("delta").mode("overwrite").saveAsTable(target_keyed))

incoming = events_keyed.limit(5000).dropDuplicates(["dedupe_key"])
incoming.createOrReplaceTempView("incoming_events_deduped")

In [0]:
%sql
MERGE INTO workspace.ecommerce.events_oct2019_keyed AS t
USING incoming_events_deduped AS s
ON t.dedupe_key = s.dedupe_key
WHEN NOT MATCHED THEN INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0
